# **Лабораторная работа №2**

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, split, lower, row_number, regexp_replace
from pyspark.sql.window import Window
import xml.etree.ElementTree as ET

spark = SparkSession.builder \
    .appName("Lab2") \
    .getOrCreate()

print(f"Версия Spark: {spark.version}")

Версия Spark: 4.0.2


# Загрузка programming_languages.csv

In [15]:
languages_df = spark.read.csv("programming-languages.csv", header=True) \
    .select(lower(col("name")).alias("lang_name")) \
    .distinct()

languages_df.show(5)

+---------+
|lang_name|
+---------+
|   ceylon|
|     hope|
|       m#|
| metafont|
|  mortran|
+---------+
only showing top 5 rows


# Загрзука posts_sample.xml

In [16]:
raw_posts = spark.sparkContext.textFile("posts_sample.xml")

def parse_xml_row(line):
    try:
        line = line.strip()
        if not line.startswith("<row"):
            return None

        root = ET.fromstring(line)
        tags = root.attrib.get("Tags", "")
        creation_date = root.attrib.get("CreationDate", "")

        if tags and creation_date:
            year = creation_date[:4]
            cleaned_tags = tags.replace("<", " ").replace(">", " ").strip()
            return (cleaned_tags, year)
    except:
        return None
    return None

parsed_posts_rdd = raw_posts.map(parse_xml_row).filter(lambda x: x is not None)
posts_df = parsed_posts_rdd.toDF(["tags", "year"])

posts_df.show(5)

+--------------------+----+
|                tags|year|
+--------------------+----+
|c#  floating-poin...|2008|
|html  css  intern...|2008|
|  c#  .net  datetime|2008|
|c#  datetime  tim...|2008|
|html  browser  ti...|2008|
+--------------------+----+
only showing top 5 rows


In [17]:
report_df = posts_df \
    .filter((col("year") >= "2010") & (col("year") <= "2020")) \
    .withColumn("tag", explode(split(col("tags"), "\\s+"))) \
    .withColumn("tag", lower(col("tag"))) \
    .join(languages_df, col("tag") == col("lang_name")) \
    .groupBy("year", "tag") \
    .count()

report_df.show(5)

+----+----------+-----+
|year|       tag|count|
+----+----------+-----+
|2013|    erlang|    3|
|2017|typescript|   20|
|2017|       sed|    2|
|2013|javascript|  198|
|2013|        f#|    6|
+----+----------+-----+
only showing top 5 rows


In [18]:
window_spec = Window.partitionBy("year").orderBy(col("count").desc())

final_report = report_df \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 10) \
    .select("year", "tag", "count") \
    .orderBy("year", col("count").desc())

# Вывод итоговой таблицы
final_report.show(200)

# Сохранение в Parquet
final_report.write.mode("overwrite").parquet("top_10_languages_report.parquet")

+----+-----------+-----+
|year|        tag|count|
+----+-----------+-----+
|2010|       java|   52|
|2010|        php|   46|
|2010| javascript|   44|
|2010|     python|   26|
|2010|objective-c|   23|
|2010|          c|   20|
|2010|       ruby|   12|
|2010|     delphi|    8|
|2010|          r|    3|
|2010|applescript|    3|
|2011|        php|  102|
|2011|       java|   93|
|2011| javascript|   83|
|2011|     python|   37|
|2011|objective-c|   34|
|2011|          c|   24|
|2011|       ruby|   20|
|2011|       perl|    9|
|2011|     delphi|    8|
|2011|       bash|    7|
|2012|        php|  154|
|2012| javascript|  132|
|2012|       java|  124|
|2012|     python|   69|
|2012|objective-c|   45|
|2012|          c|   27|
|2012|       ruby|   27|
|2012|       bash|   10|
|2012|          r|    9|
|2012|      xpath|    6|
|2013| javascript|  198|
|2013|        php|  198|
|2013|       java|  194|
|2013|     python|   90|
|2013|objective-c|   40|
|2013|          c|   36|
|2013|       ruby|   32|


In [19]:
spark.stop()